In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

import joblib
import random
import numpy as np
import scipy as sp
from matplotlib import pyplot as plt
import matplotlib as mpl
from syd import make_viewer, Viewer
from tqdm import tqdm
import optuna

from vrAnalysis.database import get_database
from vrAnalysis.helpers import Timer, get_placefield_location, uniq_val_filter
from vrAnalysis.processors.em import process_session
from vrAnalysis.processors.placefields import get_placefield, get_placefield_prediction, convert_position_to_bins
from vrAnalysis.processors.support import get_gauss_kernel, convolve_toeplitz, smooth
from dimensionality_manuscript.registry import PopulationRegistry
from dimensionality_manuscript import LocPredConfig
from dimensionality_manuscript import ResultsStore, ResultsAggregator

plt.rcParams["font.size"] = 18

# get session database
sessiondb = get_database("vrSessions")

# get population registry and models
registry = PopulationRegistry()
cfg = LocPredConfig()

In [6]:
isession = 41
spks_type = "oasis"
session_iterable = sessiondb.iter_sessions(imaging=True, session_params=dict(spks_type=spks_type))

# session = random.choice([s for s in session_iterable if s.mouse_name == "ATL027"])
session = session_iterable[isession]
print(session)
print(session.environments)

results = cfg.process(session, registry)

B2Session(mouse_name='ATL022', date='2023-04-18', session_id='701', spks_type='oasis')
[1 3]


In [4]:
results.keys()

dict_keys(['likelihood_matrix', 'loss_trajectory', 'loss_scalar', 'true_position_bins_te', 'idx_keep_rois'])

In [7]:
results["loss_scalar"]

{'poisson_cross_entropy': 5.781985282897949,
 'poisson_rank_loss_logistic_mean': 0.18853002786636353,
 'poisson_rank_metric_mean_rank': 5.114959469417833,
 'poisson_rank_metric_median_rank': 3.0,
 'poisson_rank_metric_mrr': 0.4045503138000664,
 'poisson_rank_metric_top1': 0.184966838614591,
 'poisson_rank_metric_top5': 0.7229182019159912,
 'poisson_rank_metric_top10': 0.9211495946941783,
 'poisson_distance_error': 4.100965107646622,
 'poisson_env_swap': 0.007369196757553427,
 'gaussian_cross_entropy': 3.8593130111694336,
 'gaussian_rank_loss_logistic_mean': 0.229611337184906,
 'gaussian_rank_metric_mean_rank': 4.114222549742078,
 'gaussian_rank_metric_median_rank': 3.0,
 'gaussian_rank_metric_mrr': 0.4447497183227459,
 'gaussian_rank_metric_top1': 0.2196020633750921,
 'gaussian_rank_metric_top5': 0.7848194546794399,
 'gaussian_rank_metric_top10': 0.9439941046425939,
 'gaussian_distance_error': 3.647761194029851,
 'gaussian_env_swap': 0.012527634487840826,
 'diag_gaussian_cross_entropy'